In [ ]:

import os
import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import matplotlib

SAVE_DIR = '.'
os.makedirs(SAVE_DIR, exist_ok=True)
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor':'#0d0d0d','axes.facecolor':'#0d0d0d',
    'axes.edgecolor':'#444','text.color':'white','axes.labelcolor':'white',
    'xtick.color':'white','ytick.color':'white','grid.color':'#333',
    'axes.titlecolor':'white','legend.facecolor':'#1a1a1a','legend.edgecolor':'#444'})

def dst1_t(u):
    N=u.shape[-1]; z=torch.zeros(*u.shape[:-1],1,dtype=u.dtype,device=u.device)
    return -torch.fft.rfft(torch.cat([z,u,z,u.flip(-1).neg()],dim=-1)).imag[...,1:N+1]

def idst1_t(uh): return dst1_t(uh)/(2*(uh.shape[-1]+1))
def dst1(u):  return dst1_t(torch.tensor(u,dtype=torch.float32)).numpy()
def idst1(uh): return idst1_t(torch.tensor(uh,dtype=torch.float32)).numpy()

class NLSOperator:
    def __init__(self, N, beta, L=10.0):
        self.N, self.beta, self.L = N, beta, L
        self.x_dst  = np.linspace(0,2*L,N+2,dtype=np.float32)[1:-1]
        self.x_phys = self.x_dst - L
        k = np.arange(1,N+1,dtype=np.float32)
        self.k_arr    = k
        self.lam_diff = (np.pi*k/(2*L))**2
        self.lam_k    = self.lam_diff + beta

    def phi_xx(self, phi): return idst1(-self.lam_diff*dst1(phi))
    def residual(self, phi, f): return (f-(-self.phi_xx(phi)+self.beta*phi-phi**3)).astype(np.float32)
    def make_f(self, phi): return (-self.phi_xx(phi)+self.beta*phi-phi**3).astype(np.float32)
    def inv_L(self, rhs): return idst1(dst1(rhs)/self.lam_k)
    def check(self):
        v=np.random.randn(self.N).astype(np.float32)
        return float(np.linalg.norm(self.inv_L(idst1(self.lam_k*dst1(v)))-v)/np.linalg.norm(v))

def select_frequencies(e_n, op, gd_floor=0.05, max_freqs=32):
    e_hat = dst1(e_n)
    amps  = np.abs(e_hat)
    weighted = amps / op.lam_k
    tau_w = max(2.858*np.median(weighted), gd_floor*weighted.max())
    mask  = weighted > tau_w
    if mask.sum()==0: mask[np.argmax(weighted)]=True
    if mask.sum()>max_freqs:
        idx=np.argsort(weighted)[::-1][:max_freqs]
        mask=np.zeros(op.N,dtype=bool); mask[idx]=True
    k_sel     = np.where(mask)[0]+1
    alphas    = weighted[mask]/weighted[mask].sum()  
    e_hat_sel = e_hat[mask]
    return k_sel, alphas, e_hat_sel, e_hat, mask

class FreqSubNet(nn.Module):
    def __init__(self, k_i, L, width=128, depth=3):
        super().__init__()
        self.register_buffer('k_i', torch.tensor(k_i,dtype=torch.float32))
        self.register_buffer('L',   torch.tensor(L,  dtype=torch.float32))
        self.proj  = nn.Linear(3,width)
        layers=[]
        for _ in range(depth): layers+=[nn.Linear(width,width),nn.Tanh()]
        self.trunk = nn.Sequential(*layers)
        self.head  = nn.Linear(width,1)
        self._init()

    def _init(self):
        nn.init.xavier_normal_(self.proj.weight,gain=1.0); nn.init.zeros_(self.proj.bias)
        for l in self.trunk:
            if isinstance(l,nn.Linear):
                nn.init.xavier_normal_(l.weight,gain=0.5); nn.init.zeros_(l.bias)
        nn.init.xavier_normal_(self.head.weight,gain=0.3); nn.init.zeros_(self.head.bias)

    def make_feat(self, x):
        phase=torch.tensor(np.pi,dtype=torch.float32)*self.k_i*(x+self.L)/(2*self.L)
        return torch.cat([torch.sin(phase),torch.cos(phase),x],dim=-1)

    def forward(self, x, cross_h=None):
        h=torch.tanh(self.proj(self.make_feat(x)))
        if cross_h is not None: h=torch.tanh(h+cross_h)
        h=self.trunk(h)
        return self.head(h).squeeze(-1), h

def cross_context(H, alphas_t):
    w=alphas_t.view(-1,1,1)
    return (w*H).sum(0,keepdim=True)-w*H


class SpectralDerivativeLoss(nn.Module):
    def __init__(self, N, lam_k, mask, alphas, e_hat_sel, lambda_oob=0.1):
        super().__init__()
        self.register_buffer('lam_k',   torch.tensor(lam_k,   dtype=torch.float32))
        self.register_buffer('mask_t',  torch.tensor(mask,     dtype=torch.bool))
        self.register_buffer('alpha_t', torch.tensor(alphas,   dtype=torch.float32))
        self.register_buffer('e_hat_t', torch.tensor(e_hat_sel,dtype=torch.float32))
        self.lambda_oob  = lambda_oob
        self.norm_factor = max(float((torch.tensor(alphas)*torch.tensor(e_hat_sel)**2).sum()),1e-30)

    def forward(self, g):
        g_hat   = dst1_t(g)
        Lg      = self.lam_k*g_hat
        diff    = Lg[self.mask_t]-self.e_hat_t
        loss_in = (self.alpha_t*diff**2).sum()/self.norm_factor
        loss_oob= (g_hat[~self.mask_t]**2).mean()
        return loss_in+self.lambda_oob*loss_oob, loss_in.item(), loss_oob.item()

def plot_grade_diag(op, e_n, e_hat_full, k_sel, alphas, mask,
                    losses, losses_in, losses_oob,
                    g_np, phi_before, phi_star, grade_idx, savepath):
    fig=plt.figure(figsize=(16,20))
    fig.suptitle(f'[MGDL_NLS] Grade {grade_idx}  ·  β={op.beta}  ·  '
                 f'|K_n|={len(k_sel)}',color='white',fontsize=13,y=0.99)
    gs=fig.add_gridspec(5,2,hspace=0.55,wspace=0.35,
                        left=0.07,right=0.95,top=0.96,bottom=0.04)
    k_all=op.k_arr
    amps=np.abs(e_hat_full)
    weighted=amps/op.lam_k
    tau_w=max(2.858*np.median(weighted),0.05*weighted.max())

    # Row0:
    ax=fig.add_subplot(gs[0,:])
    ax.semilogy(k_all,weighted,color='#aaaaaa',lw=0.8,label=r'$|\hat{e}_n(k)|/\lambda_k$ (weighted)')
    ax.axhline(tau_w,color='#ff4444',lw=1.5,ls='--',label=f'GD τ (weighted, floor=0.05)')
    ax.scatter(k_sel,weighted[k_sel-1],color='#00bfff',s=60,zorder=5,
               label=f'$K_n$: {len(k_sel)} active freqs')
    ax.set_title(r'$\lambda_k$-Weighted Residual Spectrum $|\hat{e}_n(k)|/\lambda_k$ — GD Threshold',color='white')
    ax.set_xlabel('k'); ax.set_ylabel('Weighted Amplitude (log)')
    ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    # Row1 left: 원시 진폭
    ax=fig.add_subplot(gs[1,0])
    ax.semilogy(k_all,amps+1e-15,color='#aaaaaa',lw=0.8,label=r'$|\hat{e}_n(k)|$')
    ax.scatter(k_sel,amps[k_sel-1],color='#ff8c00',s=50,zorder=5,label='$K_n$')
    ax.set_title('Raw Residual Spectrum',color='white')
    ax.set_xlabel('k'); ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    # Row1 right: alpha weights
    ax=fig.add_subplot(gs[1,1])
    xp=np.arange(len(k_sel))
    ax.bar(xp,alphas,color='#00e5ff',alpha=0.8,label='α_k (weighted energy)')
    ax.set_xticks(xp); ax.set_xticklabels(k_sel,fontsize=7)
    ax.set_title('α_k Energy Weights (λ_k weighted)',color='white')
    ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    # Row2 left: 
    ax=fig.add_subplot(gs[2,0])
    x_p=op.x_phys
    colors_f=plt.cm.plasma(np.linspace(0.2,0.9,min(len(k_sel),5)))
    for i,ki in enumerate(k_sel[:5]):
        phase=np.pi*ki*(x_p+op.L)/(2*op.L)
        ax.plot(x_p,np.sin(phase),color=colors_f[i],lw=1.2,label=f'k={ki}')
    ax.set_title('Input Features sin(πk(x+L)/(2L))',color='white')
    ax.set_xlabel('x'); ax.legend(fontsize=7,ncol=2); ax.grid(True,alpha=0.2)

    # Row2 right: 
    ax=fig.add_subplot(gs[2,1])
    target_g=op.inv_L(e_n)
    ax.plot(x_p,target_g,'--',color='#ffffff',lw=1.5,label='Target (inv_L·e_n)')
    ax.plot(x_p,g_np,'-',color='#b44fff',lw=1.5,label='Output g_n(x)')
    ax.set_title('Grade Output vs Target',color='white')
    ax.set_xlabel('x'); ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    # Row3 left:
    ax=fig.add_subplot(gs[3,0])
    phi_after=phi_before+g_np
    ax.plot(x_p,phi_star,'-',color='white',lw=2,label='φ*')
    ax.plot(x_p,phi_before,'--',color='#444444',lw=1.2,label='φ_{n-1}')
    ax.plot(x_p,phi_after,'-',color='#b44fff',lw=1.5,label=f'φ_{grade_idx}')
    ax.set_title('Cumulative Solution',color='white')
    ax.set_xlabel('x'); ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    # Row3 right: 
    ax=fig.add_subplot(gs[3,1])
    phi_after=phi_before+g_np
    eb=np.abs(phi_before-phi_star)+1e-15
    ea=np.abs(phi_after-phi_star)+1e-15
    rel_b=np.linalg.norm(phi_before-phi_star)/(np.linalg.norm(phi_star)+1e-30)
    rel_a=np.linalg.norm(phi_after-phi_star)/np.linalg.norm(phi_star)
    ax.semilogy(x_p,eb,'--',color='#888888',lw=1,label=f'before {rel_b:.3e}')
    ax.semilogy(x_p,ea,'-',color='#b44fff',lw=1.5,label=f'after  {rel_a:.3e}')
    ax.set_title('Pointwise Error |φ* - φ_n|',color='white')
    ax.set_xlabel('x'); ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    # Row4 left: 
    ax=fig.add_subplot(gs[4,0])
    ax.semilogy(losses,    color='#00ff99',lw=1,  label='Total')
    ax.semilogy(losses_in, '--',color='white',lw=1,label='In-Band')
    ax.semilogy(losses_oob,color='#ff4444',lw=1,  label='Out-of-Band')
    ax.set_title('Training Loss (spectral derivative)',color='white')
    ax.set_xlabel('Epoch'); ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    # Row4 right:
    ax=fig.add_subplot(gs[4,1])
    g_hat_out=dst1(g_np); target_hat=dst1(op.inv_L(e_n))
    ax.semilogy(k_all,np.abs(target_hat)+1e-15,'--',color='white',lw=1,label='target')
    ax.semilogy(k_all,np.abs(g_hat_out)+1e-15,'-',color='#b44fff',lw=1,label='output')
    for ki in k_sel: ax.axvline(ki,color='#00bfff',lw=0.5,alpha=0.4)
    ax.set_title('Output Spectrum (oob suppression)',color='white')
    ax.set_xlabel('k'); ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    plt.savefig(savepath,dpi=120,bbox_inches='tight',facecolor='#0d0d0d')
    plt.close(); print(f'  [saved] {savepath}')

def train_grade(op, e_n, phi_before, phi_star, x_t, device,
                gd_floor=0.05, width=128, depth=3,
                lambda_oob=0.1, n_epochs=2000, lr=5e-4,
                max_freqs=32, grade_idx=1):
    print(f"\n{'═'*60}\n  GRADE {grade_idx}\n{'═'*60}")
    k_sel,alphas,e_hat_sel,e_hat_full,mask=select_frequencies(e_n,op,gd_floor,max_freqs)
    m=len(k_sel)
    print(f"  |K_n|={m}  freqs={k_sel[:10]}{'...' if m>10 else ''}")
    print(f"  residual L2 = {np.linalg.norm(e_n):.4e}")

    nets=nn.ModuleList([FreqSubNet(float(k_sel[i]),op.L,width,depth).to(device) for i in range(m)])
    alphas_t=torch.tensor(alphas,dtype=torch.float32,device=device)
    criterion=SpectralDerivativeLoss(op.N,op.lam_k,mask,alphas,e_hat_sel,lambda_oob).to(device)
    params=list(nets.parameters())
    opt=optim.Adam(params,lr=lr,weight_decay=1e-6)
    sched=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=n_epochs,eta_min=lr*0.01)
    losses,losses_in,losses_oob=[],[],[]

    for ep in range(1,n_epochs+1):
        nets.train(); opt.zero_grad()
        H_list,g_list=[],[]
        for net in nets:
            g_i,h_i=net(x_t); g_list.append(g_i); H_list.append(h_i)
        H=torch.stack(H_list,dim=0); cross=cross_context(H,alphas_t)
        g2=[nets[i](x_t,cross_h=cross[i])[0] for i in range(m)]
        g_total=torch.stack(g2).sum(0)
        loss,l_in,l_oob=criterion(g_total)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params,1.0)
        opt.step(); sched.step()
        losses.append(loss.item()); losses_in.append(l_in); losses_oob.append(l_oob)
        if ep%500==0:
            print(f"  ep{ep:5d}  total={loss.item():.4e}  in={l_in:.4e}  oob={l_oob:.4e}")

    nets.eval()
    with torch.no_grad():
        H_list=[]
        for net in nets: _,h_i=net(x_t); H_list.append(h_i)
        H=torch.stack(H_list,dim=0); cross=cross_context(H,alphas_t)
        g2=[nets[i](x_t,cross_h=cross[i])[0] for i in range(m)]
        g_np=torch.stack(g2).sum(0).cpu().numpy()
    for net in nets: net.requires_grad_(False)
    print(f"  Grade {grade_idx} frozen.")
    return g_np,nets,{'k_sel':k_sel,'alphas':alphas,'n_freqs':m,
                       'losses':losses,'losses_in':losses_in,'losses_oob':losses_oob,
                       'e_hat_full':e_hat_full,'mask':mask,'g_np':g_np}


def should_stop(infos, tol=1e-5, stop_min_improve=-0.03):
    if infos[-1]['rel_err_after']<tol:
        print(f"  [STOP] rel L2 < {tol:.0e}"); return True
    if len(infos)<2: return False
    eb,ea=infos[-1]['rel_err_before'],infos[-1]['rel_err_after']
    improve=(eb-ea)/(eb+1e-30)
    pb,pa=infos[-2]['rel_err_before'],infos[-2]['rel_err_after']
    if improve<stop_min_improve or (improve<0 and (pb-pa)/(pb+1e-30)<0):
        print(f"  [EARLY STOP] improve={improve*100:.1f}%"); return True
    return False


class NLSSolverB:
    def __init__(self, op, f, phi_star,
                 gd_floor=0.05, width=128, depth=3,
                 lambda_oob=0.1, lr=5e-4, max_freqs=32,
                 stop_min_improve=-0.03, tol=1e-5, device='cpu'):
        self.op=op; self.f=f; self.phi_star=phi_star
        self.gd_floor=gd_floor; self.width=width; self.depth=depth
        self.lambda_oob=lambda_oob; self.lr=lr; self.max_freqs=max_freqs
        self.stop_min_improve=stop_min_improve; self.tol=tol
        self.device=torch.device(device)
        self._phi=np.zeros(op.N,dtype=np.float32)
        self.infos=[]; self.all_grades=[]
        self.x_t=torch.tensor(op.x_phys.reshape(-1,1),dtype=torch.float32).to(self.device)

    def solve(self, n_grades=3, n_epochs=2000):
        for gi in range(1,n_grades+1):
            e_n=self.op.residual(self._phi,self.f)
            rel_b=np.linalg.norm(self._phi-self.phi_star)/np.linalg.norm(self.phi_star)
            phi_before=self._phi.copy()
            g_np,nets,info=train_grade(
                self.op,e_n,phi_before,self.phi_star,self.x_t,self.device,
                self.gd_floor,self.width,self.depth,self.lambda_oob,
                n_epochs,self.lr,self.max_freqs,gi)
            self._phi=self._phi+g_np
            rel_a=np.linalg.norm(self._phi-self.phi_star)/np.linalg.norm(self.phi_star)
            arrow="↓" if rel_a<rel_b else "↑"
            print(f"\n  Rel L2 before : {rel_b:.4e}")
            print(f"  Rel L2 after  : {rel_a:.4e}  ({arrow}{abs(rel_a-rel_b)/(rel_b+1e-30)*100:.1f}%)")
            info['rel_err_before']=rel_b; info['rel_err_after']=rel_a
            self.infos.append(info); self.all_grades.append(nets)
            plot_grade_diag(self.op,e_n,info['e_hat_full'],info['k_sel'],
                info['alphas'],info['mask'],info['losses'],info['losses_in'],
                info['losses_oob'],g_np,phi_before,self.phi_star,gi,
                os.path.join(SAVE_DIR, f'grade_{gi:02d}_diag.png'))
            if should_stop(self.infos,self.tol,self.stop_min_improve): break
        self._table(); self._plot_summary()

    def _table(self):
        print(f"\n{'='*70}")
        print(f"{'G':^3}  {'|Kn|':^5}  {'err_before':^11}  {'err_after':^11}  {'Δ%':^8}")
        print("-"*70)
        for i,info in enumerate(self.infos):
            eb,ea=info['rel_err_before'],info['rel_err_after']
            print(f" {i+1:^3}  {info['n_freqs']:^5}  {eb:^11.4e}  {ea:^11.4e}  {(eb-ea)/(eb+1e-30)*100:^+7.1f}%")
        print("="*70)
        print(f"Final rel L2: {self.infos[-1]['rel_err_after']:.4e}")

    def _plot_summary(self):
        n=len(self.infos)
        fig,axes=plt.subplots(1,3,figsize=(15,4))
        errs=[info['rel_err_after'] for info in self.infos]
        axes[0].semilogy(range(1,n+1),errs,'o-',color='#00bfff',lw=2)
        axes[0].set_xlabel('Grade'); axes[0].set_ylabel('Rel L2')
        axes[0].set_title('Convergence'); axes[0].grid(True,alpha=0.3)
        last=self.infos[-1]
        axes[1].semilogy(last['losses'],    color='#00ff99',lw=1,label='Total')
        axes[1].semilogy(last['losses_in'], '--',color='white',lw=1,label='In-Band')
        axes[1].semilogy(last['losses_oob'],color='#ff4444',lw=1,label='Out-of-Band')
        axes[1].set_xlabel('Epoch'); axes[1].set_title(f'Grade {n} Loss')
        axes[1].legend(); axes[1].grid(True,alpha=0.3)
        axes[2].plot(self.op.x_phys,self.phi_star,'w-',lw=2,label='true')
        axes[2].plot(self.op.x_phys,self._phi,'--',color='#b44fff',lw=1.5,label='approx')
        axes[2].set_title(f'Solution (final rel={errs[-1]:.2e})')
        axes[2].legend(); axes[2].grid(True,alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(SAVE_DIR,'summary_nls.png'),dpi=150,bbox_inches='tight',facecolor='#0d0d0d')
        print(f"  [saved] {os.path.join(SAVE_DIR,'summary_nls.png')}")

if __name__=='__main__':
    device='cuda' if torch.cuda.is_available() else 'cpu'
    print(f"[*] device: {device}")
    N=256; beta=1.0; L=10.0
    op=NLSOperator(N,beta,L)
    phi_star=(0.3*np.sin(np.pi*3*op.x_dst/(2*L))+
              0.2*np.sin(np.pi*7*op.x_dst/(2*L))+
              0.1*np.sin(np.pi*11*op.x_dst/(2*L))+
              0.15*np.sin(np.pi*23*op.x_dst/(2*L))).astype(np.float32)
    f=op.make_f(phi_star)
    print(f"[check] inv_L err={op.check():.2e}")
    solver=NLSSolverB(op=op,f=f,phi_star=phi_star,
        gd_floor=0.05,width=128,depth=3,lambda_oob=0.1,
        lr=5e-4,max_freqs=32,stop_min_improve=-0.03,
        tol=1e-5,device=device)
    solver.solve(n_grades=3,n_epochs=2000)

In [ ]:

import os
import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import matplotlib

#
SAVE_DIR = '.'
os.makedirs(SAVE_DIR, exist_ok=True)
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor':'#0d0d0d','axes.facecolor':'#0d0d0d',
    'axes.edgecolor':'#444','text.color':'white','axes.labelcolor':'white',
    'xtick.color':'white','ytick.color':'white','grid.color':'#333',
    'axes.titlecolor':'white','legend.facecolor':'#1a1a1a','legend.edgecolor':'#444'})

# DST 
def dst1_t(u):
    N=u.shape[-1]; z=torch.zeros(*u.shape[:-1],1,dtype=u.dtype,device=u.device)
    return -torch.fft.rfft(torch.cat([z,u,z,u.flip(-1).neg()],dim=-1)).imag[...,1:N+1]

def idst1_t(uh): return dst1_t(uh)/(2*(uh.shape[-1]+1))
def dst1(u):  return dst1_t(torch.tensor(u,dtype=torch.float32)).numpy()
def idst1(uh): return idst1_t(torch.tensor(uh,dtype=torch.float32)).numpy()

# NLS
class NLSOperator:
    def __init__(self, N, beta, L=10.0):
        self.N, self.beta, self.L = N, beta, L
        self.x_dst  = np.linspace(0,2*L,N+2,dtype=np.float32)[1:-1]
        self.x_phys = self.x_dst - L
        k = np.arange(1,N+1,dtype=np.float32)
        self.k_arr    = k
        self.lam_diff = (np.pi*k/(2*L))**2
        self.lam_k    = self.lam_diff + beta

    def phi_xx(self, phi): return idst1(-self.lam_diff*dst1(phi))
    def residual(self, phi, f): return (f-(-self.phi_xx(phi)+self.beta*phi-phi**3)).astype(np.float32)
    def make_f(self, phi): return (-self.phi_xx(phi)+self.beta*phi-phi**3).astype(np.float32)
    def inv_L(self, rhs): return idst1(dst1(rhs)/self.lam_k)
    def check(self):
        v=np.random.randn(self.N).astype(np.float32)
        return float(np.linalg.norm(self.inv_L(idst1(self.lam_k*dst1(v)))-v)/np.linalg.norm(v))


# 3. λ_k, G-D 
def select_frequencies(e_n, op, gd_floor=0.05, max_freqs=32):
    e_hat = dst1(e_n)
    amps  = np.abs(e_hat)
    weighted = amps / op.lam_k
    tau_w = max(2.858*np.median(weighted), gd_floor*weighted.max())
    mask  = weighted > tau_w
    if mask.sum()==0: mask[np.argmax(weighted)]=True
    if mask.sum()>max_freqs:
        idx=np.argsort(weighted)[::-1][:max_freqs]
        mask=np.zeros(op.N,dtype=bool); mask[idx]=True
    k_sel     = np.where(mask)[0]+1
    alphas    = weighted[mask]/weighted[mask].sum()  # 가중 정규화
    e_hat_sel = e_hat[mask]
    return k_sel, alphas, e_hat_sel, e_hat, mask

#
class FreqSubNet(nn.Module):
    def __init__(self, k_i, L, width=128, depth=3):
        super().__init__()
        self.register_buffer('k_i', torch.tensor(k_i,dtype=torch.float32))
        self.register_buffer('L',   torch.tensor(L,  dtype=torch.float32))
        self.proj  = nn.Linear(3,width)
        layers=[]
        for _ in range(depth): layers+=[nn.Linear(width,width),nn.Tanh()]
        self.trunk = nn.Sequential(*layers)
        self.head  = nn.Linear(width,1)
        self._init()

    def _init(self):
        nn.init.xavier_normal_(self.proj.weight,gain=1.0); nn.init.zeros_(self.proj.bias)
        for l in self.trunk:
            if isinstance(l,nn.Linear):
                nn.init.xavier_normal_(l.weight,gain=0.5); nn.init.zeros_(l.bias)
        nn.init.xavier_normal_(self.head.weight,gain=0.3); nn.init.zeros_(self.head.bias)

    def make_feat(self, x):
        phase=torch.tensor(np.pi,dtype=torch.float32)*self.k_i*(x+self.L)/(2*self.L)
        return torch.cat([torch.sin(phase),torch.cos(phase),x],dim=-1)

    def forward(self, x, cross_h=None):
        h=torch.tanh(self.proj(self.make_feat(x)))
        if cross_h is not None: h=torch.tanh(h+cross_h)
        h=self.trunk(h)
        return self.head(h).squeeze(-1), h

def cross_context(H, alphas_t):
    w=alphas_t.view(-1,1,1)
    return (w*H).sum(0,keepdim=True)-w*H

# 
class SpectralDerivativeLoss(nn.Module):
    def __init__(self, N, lam_k, mask, alphas, e_hat_sel, lambda_oob=0.1):
        super().__init__()
        self.register_buffer('lam_k',   torch.tensor(lam_k,   dtype=torch.float32))
        self.register_buffer('mask_t',  torch.tensor(mask,     dtype=torch.bool))
        self.register_buffer('alpha_t', torch.tensor(alphas,   dtype=torch.float32))
        self.register_buffer('e_hat_t', torch.tensor(e_hat_sel,dtype=torch.float32))
        self.lambda_oob  = lambda_oob
        self.norm_factor = max(float((torch.tensor(alphas)*torch.tensor(e_hat_sel)**2).sum()),1e-30)

    def forward(self, g):
        g_hat   = dst1_t(g)
        Lg      = self.lam_k*g_hat
        diff    = Lg[self.mask_t]-self.e_hat_t
        loss_in = (self.alpha_t*diff**2).sum()/self.norm_factor
        loss_oob= (g_hat[~self.mask_t]**2).mean()
        return loss_in+self.lambda_oob*loss_oob, loss_in.item(), loss_oob.item()

# 
def plot_grade_diag(op, e_n, e_hat_full, k_sel, alphas, mask,
                    losses, losses_in, losses_oob,
                    g_np, phi_before, phi_star, grade_idx, savepath):
    fig=plt.figure(figsize=(16,20))
    fig.suptitle(f'[MGDL_NLS] Grade {grade_idx}  ·  β={op.beta}  ·  '
                 f'|K_n|={len(k_sel)}',color='white',fontsize=13,y=0.99)
    gs=fig.add_gridspec(5,2,hspace=0.55,wspace=0.35,
                        left=0.07,right=0.95,top=0.96,bottom=0.04)
    k_all=op.k_arr
    amps=np.abs(e_hat_full)
    weighted=amps/op.lam_k
    tau_w=max(2.858*np.median(weighted),0.05*weighted.max())

    # Row0: 
    ax=fig.add_subplot(gs[0,:])
    ax.semilogy(k_all,weighted,color='#aaaaaa',lw=0.8,label=r'$|\hat{e}_n(k)|/\lambda_k$ (weighted)')
    ax.axhline(tau_w,color='#ff4444',lw=1.5,ls='--',label=f'GD τ (weighted, floor=0.05)')
    ax.scatter(k_sel,weighted[k_sel-1],color='#00bfff',s=60,zorder=5,
               label=f'$K_n$: {len(k_sel)} active freqs')
    ax.set_title(r'$\lambda_k$-Weighted Residual Spectrum $|\hat{e}_n(k)|/\lambda_k$ — GD Threshold',color='white')
    ax.set_xlabel('k'); ax.set_ylabel('Weighted Amplitude (log)')
    ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    # Row1 left: 
    ax=fig.add_subplot(gs[1,0])
    ax.semilogy(k_all,amps+1e-15,color='#aaaaaa',lw=0.8,label=r'$|\hat{e}_n(k)|$')
    ax.scatter(k_sel,amps[k_sel-1],color='#ff8c00',s=50,zorder=5,label='$K_n$')
    ax.set_title('Raw Residual Spectrum',color='white')
    ax.set_xlabel('k'); ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    # Row1 right: alpha weights
    ax=fig.add_subplot(gs[1,1])
    xp=np.arange(len(k_sel))
    ax.bar(xp,alphas,color='#00e5ff',alpha=0.8,label='α_k (weighted energy)')
    ax.set_xticks(xp); ax.set_xticklabels(k_sel,fontsize=7)
    ax.set_title('α_k Energy Weights (λ_k weighted)',color='white')
    ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    # Row2 left:
    ax=fig.add_subplot(gs[2,0])
    x_p=op.x_phys
    colors_f=plt.cm.plasma(np.linspace(0.2,0.9,min(len(k_sel),5)))
    for i,ki in enumerate(k_sel[:5]):
        phase=np.pi*ki*(x_p+op.L)/(2*op.L)
        ax.plot(x_p,np.sin(phase),color=colors_f[i],lw=1.2,label=f'k={ki}')
    ax.set_title('Input Features sin(πk(x+L)/(2L))',color='white')
    ax.set_xlabel('x'); ax.legend(fontsize=7,ncol=2); ax.grid(True,alpha=0.2)

    # Row2 right: 
    ax=fig.add_subplot(gs[2,1])
    target_g=op.inv_L(e_n)
    ax.plot(x_p,target_g,'--',color='#ffffff',lw=1.5,label='Target (inv_L·e_n)')
    ax.plot(x_p,g_np,'-',color='#b44fff',lw=1.5,label='Output g_n(x)')
    ax.set_title('Grade Output vs Target',color='white')
    ax.set_xlabel('x'); ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    # Row3 left: 
    ax=fig.add_subplot(gs[3,0])
    phi_after=phi_before+g_np
    ax.plot(x_p,phi_star,'-',color='white',lw=2,label='φ*')
    ax.plot(x_p,phi_before,'--',color='#444444',lw=1.2,label='φ_{n-1}')
    ax.plot(x_p,phi_after,'-',color='#b44fff',lw=1.5,label=f'φ_{grade_idx}')
    ax.set_title('Cumulative Solution',color='white')
    ax.set_xlabel('x'); ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    # Row3 right: 
    ax=fig.add_subplot(gs[3,1])
    phi_after=phi_before+g_np
    eb=np.abs(phi_before-phi_star)+1e-15
    ea=np.abs(phi_after-phi_star)+1e-15
    rel_b=np.linalg.norm(phi_before-phi_star)/(np.linalg.norm(phi_star)+1e-30)
    rel_a=np.linalg.norm(phi_after-phi_star)/np.linalg.norm(phi_star)
    ax.semilogy(x_p,eb,'--',color='#888888',lw=1,label=f'before {rel_b:.3e}')
    ax.semilogy(x_p,ea,'-',color='#b44fff',lw=1.5,label=f'after  {rel_a:.3e}')
    ax.set_title('Pointwise Error |φ* - φ_n|',color='white')
    ax.set_xlabel('x'); ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    # Row4 left: 
    ax=fig.add_subplot(gs[4,0])
    ax.semilogy(losses,    color='#00ff99',lw=1,  label='Total')
    ax.semilogy(losses_in, '--',color='white',lw=1,label='In-Band')
    ax.semilogy(losses_oob,color='#ff4444',lw=1,  label='Out-of-Band')
    ax.set_title('Training Loss (spectral derivative)',color='white')
    ax.set_xlabel('Epoch'); ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    # Row4 right: 
    ax=fig.add_subplot(gs[4,1])
    g_hat_out=dst1(g_np); target_hat=dst1(op.inv_L(e_n))
    ax.semilogy(k_all,np.abs(target_hat)+1e-15,'--',color='white',lw=1,label='target')
    ax.semilogy(k_all,np.abs(g_hat_out)+1e-15,'-',color='#b44fff',lw=1,label='output')
    for ki in k_sel: ax.axvline(ki,color='#00bfff',lw=0.5,alpha=0.4)
    ax.set_title('Output Spectrum (oob suppression)',color='white')
    ax.set_xlabel('k'); ax.legend(fontsize=8); ax.grid(True,alpha=0.2)

    plt.savefig(savepath,dpi=120,bbox_inches='tight',facecolor='#0d0d0d')
    plt.close(); print(f'  [saved] {savepath}')

# 
def train_grade(op, e_n, phi_before, phi_star, x_t, device,
                gd_floor=0.05, width=128, depth=3,
                lambda_oob=0.1, n_epochs=2000, lr=5e-4,
                max_freqs=32, grade_idx=1):
    print(f"\n{'═'*60}\n  GRADE {grade_idx}\n{'═'*60}")
    k_sel,alphas,e_hat_sel,e_hat_full,mask=select_frequencies(e_n,op,gd_floor,max_freqs)
    m=len(k_sel)
    print(f"  |K_n|={m}  freqs={k_sel[:10]}{'...' if m>10 else ''}")
    print(f"  residual L2 = {np.linalg.norm(e_n):.4e}")

    nets=nn.ModuleList([FreqSubNet(float(k_sel[i]),op.L,width,depth).to(device) for i in range(m)])
    alphas_t=torch.tensor(alphas,dtype=torch.float32,device=device)
    criterion=SpectralDerivativeLoss(op.N,op.lam_k,mask,alphas,e_hat_sel,lambda_oob).to(device)
    params=list(nets.parameters())
    opt=optim.Adam(params,lr=lr,weight_decay=1e-6)
    sched=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=n_epochs,eta_min=lr*0.01)
    losses,losses_in,losses_oob=[],[],[]

    for ep in range(1,n_epochs+1):
        nets.train(); opt.zero_grad()
        H_list,g_list=[],[]
        for net in nets:
            g_i,h_i=net(x_t); g_list.append(g_i); H_list.append(h_i)
        H=torch.stack(H_list,dim=0); cross=cross_context(H,alphas_t)
        g2=[nets[i](x_t,cross_h=cross[i])[0] for i in range(m)]
        g_total=torch.stack(g2).sum(0)
        loss,l_in,l_oob=criterion(g_total)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params,1.0)
        opt.step(); sched.step()
        losses.append(loss.item()); losses_in.append(l_in); losses_oob.append(l_oob)
        if ep%500==0:
            print(f"  ep{ep:5d}  total={loss.item():.4e}  in={l_in:.4e}  oob={l_oob:.4e}")

    nets.eval()
    with torch.no_grad():
        H_list=[]
        for net in nets: _,h_i=net(x_t); H_list.append(h_i)
        H=torch.stack(H_list,dim=0); cross=cross_context(H,alphas_t)
        g2=[nets[i](x_t,cross_h=cross[i])[0] for i in range(m)]
        g_np=torch.stack(g2).sum(0).cpu().numpy()
    for net in nets: net.requires_grad_(False)
    print(f"  Grade {grade_idx} frozen.")
    return g_np,nets,{'k_sel':k_sel,'alphas':alphas,'n_freqs':m,
                       'losses':losses,'losses_in':losses_in,'losses_oob':losses_oob,
                       'e_hat_full':e_hat_full,'mask':mask,'g_np':g_np}

# 
def should_stop(infos, tol=1e-5, stop_min_improve=-0.03):
    if infos[-1]['rel_err_after']<tol:
        print(f"  [STOP] rel L2 < {tol:.0e}"); return True
    if len(infos)<2: return False
    eb,ea=infos[-1]['rel_err_before'],infos[-1]['rel_err_after']
    improve=(eb-ea)/(eb+1e-30)
    pb,pa=infos[-2]['rel_err_before'],infos[-2]['rel_err_after']
    if improve<stop_min_improve or (improve<0 and (pb-pa)/(pb+1e-30)<0):
        print(f"  [EARLY STOP] improve={improve*100:.1f}%"); return True
    return False

# 
class NLSSolverB:
    def __init__(self, op, f, phi_star,
                 gd_floor=0.05, width=128, depth=3,
                 lambda_oob=0.1, lr=5e-4, max_freqs=32,
                 stop_min_improve=-0.03, tol=1e-5, device='cpu'):
        self.op=op; self.f=f; self.phi_star=phi_star
        self.gd_floor=gd_floor; self.width=width; self.depth=depth
        self.lambda_oob=lambda_oob; self.lr=lr; self.max_freqs=max_freqs
        self.stop_min_improve=stop_min_improve; self.tol=tol
        self.device=torch.device(device)
        self._phi=np.zeros(op.N,dtype=np.float32)
        self.infos=[]; self.all_grades=[]
        self.x_t=torch.tensor(op.x_phys.reshape(-1,1),dtype=torch.float32).to(self.device)

    def solve(self, n_grades=3, n_epochs=2000):
        for gi in range(1,n_grades+1):
            e_n=self.op.residual(self._phi,self.f)
            rel_b=np.linalg.norm(self._phi-self.phi_star)/np.linalg.norm(self.phi_star)
            phi_before=self._phi.copy()
            g_np,nets,info=train_grade(
                self.op,e_n,phi_before,self.phi_star,self.x_t,self.device,
                self.gd_floor,self.width,self.depth,self.lambda_oob,
                n_epochs,self.lr,self.max_freqs,gi)
            self._phi=self._phi+g_np
            rel_a=np.linalg.norm(self._phi-self.phi_star)/np.linalg.norm(self.phi_star)
            arrow="↓" if rel_a<rel_b else "↑"
            print(f"\n  Rel L2 before : {rel_b:.4e}")
            print(f"  Rel L2 after  : {rel_a:.4e}  ({arrow}{abs(rel_a-rel_b)/(rel_b+1e-30)*100:.1f}%)")
            info['rel_err_before']=rel_b; info['rel_err_after']=rel_a
            self.infos.append(info); self.all_grades.append(nets)
            plot_grade_diag(self.op,e_n,info['e_hat_full'],info['k_sel'],
                info['alphas'],info['mask'],info['losses'],info['losses_in'],
                info['losses_oob'],g_np,phi_before,self.phi_star,gi,
                os.path.join(SAVE_DIR, f'grade_{gi:02d}_diag.png'))
            if should_stop(self.infos,self.tol,self.stop_min_improve): break
        self._table(); self._plot_summary()

    def _table(self):
        print(f"\n{'='*70}")
        print(f"{'G':^3}  {'|Kn|':^5}  {'err_before':^11}  {'err_after':^11}  {'Δ%':^8}")
        print("-"*70)
        for i,info in enumerate(self.infos):
            eb,ea=info['rel_err_before'],info['rel_err_after']
            print(f" {i+1:^3}  {info['n_freqs']:^5}  {eb:^11.4e}  {ea:^11.4e}  {(eb-ea)/(eb+1e-30)*100:^+7.1f}%")
        print("="*70)
        print(f"Final rel L2: {self.infos[-1]['rel_err_after']:.4e}")

    def _plot_summary(self):
        n = len(self.infos)
        errs = [info['rel_err_after'] for info in self.infos]
        cmap   = plt.cm.plasma
        colors = [cmap(i/(max(n-1,1))) for i in range(n)]
        from matplotlib.lines import Line2D

        fig = plt.figure(figsize=(22, 16))
        gs  = fig.add_gridspec(3, 3, hspace=0.50, wspace=0.35,
                               left=0.07, right=0.97, top=0.93, bottom=0.07)

        # ── Row0: Continuous loss plot across three columns──
        ax_loss = fig.add_subplot(gs[0, :])
        offset  = 0
        xticks_pos, xticks_lab = [], []
        for i, info in enumerate(self.infos):
            ne = len(info['losses'])
            ep = list(range(offset + 1, offset + ne + 1))
            ax_loss.semilogy(ep, info['losses'],     color=colors[i], lw=1.5,  label=f'G{i+1}')
            ax_loss.semilogy(ep, info['losses_in'],  color=colors[i], lw=0.9,  ls='--', alpha=0.8)
            ax_loss.semilogy(ep, info['losses_oob'], color=colors[i], lw=0.9,  ls=':',  alpha=0.6)
            if i > 0:
                ax_loss.axvline(offset, color='#555555', lw=1.0, ls='--')
            xticks_pos.append(offset + ne // 2)
            xticks_lab.append(f'G{i+1}')
            offset += ne
        leg = [Line2D([0],[0], color='white', lw=1.5,        label='Total'),
               Line2D([0],[0], color='white', lw=0.9, ls='--', label='In-Band'),
               Line2D([0],[0], color='white', lw=0.9, ls=':',  label='OOB')]
        ax_loss.legend(handles=leg, fontsize=9, loc='upper right')
        ax_loss.set_xticks(xticks_pos); ax_loss.set_xticklabels(xticks_lab, fontsize=9)
        ax_loss.set_xlabel(f'Epoch  (each grade = {len(self.infos[0]["losses"])} epochs)')
        ax_loss.set_ylabel('Loss')
        ax_loss.set_title('All Grades — Continuous Training Loss  (solid=Total · dashed=In-Band · dotted=OOB)', color='white')
        ax_loss.set_xlim(1, offset)
        ax_loss.grid(True, alpha=0.2)

        # 
        ax = fig.add_subplot(gs[1, 0])
        ax.plot(self.op.x_phys, self.phi_star, 'w-',  lw=2,   label='φ* (true)')
        ax.plot(self.op.x_phys, self._phi,     '--',  color='#b44fff', lw=1.5,
                label=f'φ_approx (rel={errs[-1]:.2e})')
        ax.set_title('Final Solution vs True', color='white')
        ax.set_xlabel('x'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

        # 
        ax = fig.add_subplot(gs[1, 1])
        phi_cum = np.zeros(self.op.N, dtype=np.float32)
        ax.plot(self.op.x_phys, self.phi_star, 'w-', lw=2, label='φ*', zorder=10)
        for i, info in enumerate(self.infos):
            phi_cum = phi_cum + info['g_np']
            alpha_v = 0.3 + 0.7*(i+1)/n
            ax.plot(self.op.x_phys, phi_cum, color=colors[i],
                    lw=1.0, alpha=alpha_v, label=f'G{i+1}')
        ax.set_title('Solution Evolution per Grade', color='white')
        ax.set_xlabel('x'); ax.legend(fontsize=7, ncol=4); ax.grid(True, alpha=0.3)

        # 
        ax = fig.add_subplot(gs[1, 2])
        phi_cum = np.zeros(self.op.N, dtype=np.float32)
        for i, info in enumerate(self.infos):
            phi_cum = phi_cum + info['g_np']
            err = np.abs(phi_cum - self.phi_star) + 1e-15
            ax.semilogy(self.op.x_phys, err, color=colors[i],
                        lw=1.0, alpha=0.8, label=f'G{i+1}')
        ax.set_title('Pointwise Error Evolution', color='white')
        ax.set_xlabel('x'); ax.legend(fontsize=7, ncol=4); ax.grid(True, alpha=0.3)

        # 
        ax = fig.add_subplot(gs[2, 0])
        ax.semilogy(range(1, n+1), errs, 'o-', color='#00bfff', lw=2, ms=6)
        for i, e in enumerate(errs):
            ax.annotate(f'{e:.1e}', (i+1, e), textcoords='offset points',
                        xytext=(4, 4), fontsize=7, color='#aaaaaa')
        ax.set_xlabel('Grade'); ax.set_ylabel('Rel L2')
        ax.set_title('Rel L2 Convergence per Grade', color='white')
        ax.set_xticks(range(1, n+1)); ax.grid(True, alpha=0.3)

        # |K_n|
        ax = fig.add_subplot(gs[2, 1])
        kns = [info['n_freqs'] for info in self.infos]
        bars = ax.bar(range(1, n+1), kns, color=colors, alpha=0.85)
        for bar, k in zip(bars, kns):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                    str(k), ha='center', va='bottom', fontsize=8, color='white')
        ax.set_xlabel('Grade'); ax.set_ylabel('|K_n|')
        ax.set_title('Active Frequencies per Grade', color='white')
        ax.set_xticks(range(1, n+1)); ax.grid(True, alpha=0.2, axis='y')

        # 
        ax = fig.add_subplot(gs[2, 2])
        improves = [(info['rel_err_before']-info['rel_err_after'])/
                    (info['rel_err_before']+1e-30)*100
                    for info in self.infos]
        bar_colors = ['#00ff99' if v>0 else '#ff4444' for v in improves]
        ax.bar(range(1, n+1), improves, color=bar_colors, alpha=0.85)
        for i, v in enumerate(improves):
            ax.text(i+1, v+0.5, f'{v:.1f}%', ha='center', va='bottom',
                    fontsize=7, color='white')
        ax.axhline(0, color='white', lw=0.8)
        ax.set_xlabel('Grade'); ax.set_ylabel('Improvement (%)')
        ax.set_title('Rel L2 Improvement per Grade', color='white')
        ax.set_xticks(range(1, n+1)); ax.grid(True, alpha=0.2, axis='y')

        fig.suptitle(
            f'[MGDL_NLS] Summary  ·  β={self.op.beta}  ·  {n} Grades  ·  '
            f'Final rel L2={errs[-1]:.2e}',
            color='white', fontsize=13)

        savepath = os.path.join(SAVE_DIR, 'summary_nls.png')
        plt.savefig(savepath, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
        print(f"  [saved] {savepath}")

# 
if __name__=='__main__':
    device='cuda' if torch.cuda.is_available() else 'cpu'
    print(f"[*] device: {device}")
    N=256; beta=1.0; L=10.0
    op=NLSOperator(N,beta,L)
    phi_star=(0.3*np.sin(np.pi*3*op.x_dst/(2*L))+
              0.2*np.sin(np.pi*7*op.x_dst/(2*L))+
              0.1*np.sin(np.pi*11*op.x_dst/(2*L))+
              0.15*np.sin(np.pi*23*op.x_dst/(2*L))).astype(np.float32)
    f=op.make_f(phi_star)
    print(f"[check] inv_L err={op.check():.2e}")
    solver=NLSSolverB(op=op,f=f,phi_star=phi_star,
        gd_floor=0.05,width=128,depth=3,lambda_oob=0.1,
        lr=5e-4,max_freqs=32,stop_min_improve=-0.03,
        tol=1e-5,device=device)
    solver.solve(n_grades=3,n_epochs=2000)